TALLER 3

Integrantes:
- Cristóbal Salvo
- Fernando Núñez
- Fabian Mejías
- Cristóbal Valenzuela

Asignatura: Ciencia de Datos (NRC: 7931)

In [6]:
############################################
### NO ES NECESARIO EJECUTAR ESTE CODIGO ###
############################################

# Instalación de Apache Spark en Google Colab
#!apt-get install openjdk-8-jdk-headless -qq > /dev/null
#!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
#!tar xf spark-3.5.0-bin-hadoop3.tgz
#!pip install -q findspark

#import os
# Configuramos las variables de entorno
#os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
#os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"

#import findspark
#findspark.init()


import os
import sys
from pyspark.sql import SparkSession

# Configuración para evitar conflictos de versiones de Python en local
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Crear sesión de Spark
spark = SparkSession.builder \
    .appName("AnalisisEgresos2024") \
    .getOrCreate()


Error: Se ha producido un error de enlace al cargar la clase principal org.apache.spark.launcher.Main
	java.lang.UnsupportedClassVersionError: org/apache/spark/launcher/Main has been compiled by a more recent version of the Java Runtime (class file version 61.0), this version of the Java Runtime only recognizes class file versions up to 55.0
/home/fernando/Herramientas/anaconda3/envs/csd_proyecto/lib/python3.10/site-packages/pyspark/bin/spark-class: línea 97: CMD: subíndice de matriz incorrecto


PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AnalisisEgresos2024") \
    .getOrCreate()

# TIENEN QUE PEGAR EL ARCHIVO EGRESOS_2024.csv EN EL LOCAL DEL COLAB
df = spark.read.csv(
    './data/EGRESOS_2024.csv',
    header=True,
    inferSchema=True,
    sep=';'
)

df.show(5)

+---------------------------------+----+------------+-----------------+-----------------+-----------------------+-----------------+-----------------------+---------+---------------+----------+-----+-----+-----------+----------------+
|PERTENENCIA_ESTABLECIMIENTO_SALUD|SEXO|  GRUPO_EDAD|GLOSA_PAIS_ORIGEN|COMUNA_RESIDENCIA|GLOSA_COMUNA_RESIDENCIA|REGION_RESIDENCIA|GLOSA_REGION_RESIDENCIA|PREVISION|GLOSA_PREVISION|ANO_EGRESO|DIAG1|DIAG2|DIAS_ESTADA|CONDICION_EGRESO|
+---------------------------------+----+------------+-----------------+-----------------+-----------------------+-----------------+-----------------------+---------+---------------+----------+-----+-----+-----------+----------------+
|             Pertenecientes al...|   2|20 A 24 A�OS|        Argentina|            01101|                Iquique|               01|            De Tarapac�|        1|         FONASA|      2024| N908| NULL|          2|               1|
|             Pertenecientes al...|   2|25 A 29 A�OS|        Arg

In [ ]:
#########################
### Analisis de datos ###
#########################

In [ ]:
# Ver la estructura del dataset y los tipos de datos inferidos
df.printSchema()

root
 |-- PERTENENCIA_ESTABLECIMIENTO_SALUD: string (nullable = true)
 |-- SEXO: string (nullable = true)
 |-- GRUPO_EDAD: string (nullable = true)
 |-- GLOSA_PAIS_ORIGEN: string (nullable = true)
 |-- COMUNA_RESIDENCIA: string (nullable = true)
 |-- GLOSA_COMUNA_RESIDENCIA: string (nullable = true)
 |-- REGION_RESIDENCIA: string (nullable = true)
 |-- GLOSA_REGION_RESIDENCIA: string (nullable = true)
 |-- PREVISION: string (nullable = true)
 |-- GLOSA_PREVISION: string (nullable = true)
 |-- ANO_EGRESO: integer (nullable = true)
 |-- DIAG1: string (nullable = true)
 |-- DIAG2: string (nullable = true)
 |-- DIAS_ESTADA: integer (nullable = true)
 |-- CONDICION_EGRESO: integer (nullable = true)



-------------

In [ ]:
# Ver la distribución, cuartiles y posibles valores atípicos
df.select("DIAS_ESTADA").summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

+-------+------------------+
|summary|       DIAS_ESTADA|
+-------+------------------+
|  count|           1667349|
|   mean|  6.09228781736757|
| stddev|40.362834562407244|
|    min|                 1|
|    25%|                 1|
|    50%|                 2|
|    75%|                 6|
|    max|             17858|
+-------+------------------+



In [ ]:
from pyspark.sql.functions import count, when, isnan, col

# Buscar nulos, celdas vacías o el string "NULL" en todas las columnas
df.select([
    count(when(isnan(c) | col(c).isNull() | (col(c) == "") | (col(c) == "NULL") | (col(c) == "Desconocido"), c)).alias(c)
    for c in df.columns
]).show(vertical=True)

-RECORD 0------------------------------------
 PERTENENCIA_ESTABLECIMIENTO_SALUD | 0       
 SEXO                              | 0       
 GRUPO_EDAD                        | 0       
 GLOSA_PAIS_ORIGEN                 | 0       
 COMUNA_RESIDENCIA                 | 0       
 GLOSA_COMUNA_RESIDENCIA           | 0       
 REGION_RESIDENCIA                 | 912     
 GLOSA_REGION_RESIDENCIA           | 0       
 PREVISION                         | 0       
 GLOSA_PREVISION                   | 0       
 ANO_EGRESO                        | 0       
 DIAG1                             | 0       
 DIAG2                             | 1506612 
 DIAS_ESTADA                       | 0       
 CONDICION_EGRESO                  | 0       



In [ ]:
# Revisar valores únicos en SEXO (¿hay algo más aparte de 1 y 2?)
df.groupBy("SEXO").count().orderBy("count").show()

# Revisar los grupos de edad (para ver si hay formatos inconsistentes)
df.groupBy("GRUPO_EDAD").count().orderBy("GRUPO_EDAD").show(30, truncate=False)

# Revisar Previsión
df.groupBy("GLOSA_PREVISION").count().orderBy("count").show(truncate=False)

+----+------+
|SEXO| count|
+----+------+
|   *| 28382|
|   1|704881|
|   2|934086|
+----+------+

+------------------------+------+
|GRUPO_EDAD              |count |
+------------------------+------+
|1 A 4 A�OS              |53829 |
|10 A 14 A�OS            |46948 |
|15 A 19 A�OS            |54348 |
|2 MESES A MENOS DE 1 A�O|18914 |
|20 A 24 A�OS            |80340 |
|25 A 29 A�OS            |110946|
|28 DIAS A 2 MES         |3984  |
|30 A 34 A�OS            |137464|
|35 A 39 A�OS            |125138|
|40 A 44 A�OS            |99615 |
|45 A 49 A�OS            |87462 |
|5 A 9 A�OS              |52588 |
|50 A 54 A�OS            |91040 |
|55 A 59 A�OS            |102125|
|60 A 64 A�OS            |115953|
|65 A 69 A�OS            |116720|
|7 A 27 DIAS             |5281  |
|70 A 74 A�OS            |104227|
|75 A 79 A�OS            |92283 |
|80 A 84 A�OS            |69633 |
|85 A MAS                |71185 |
|menor a 7 d�as          |27326 |
+------------------------+------+

+---------------

In [ ]:
##############################
### Procesamiento de datos ###
##############################

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col, substring, when, lit
import pandas as pd

SEED = 256
FRACTION_SAMPLE = 0.1
CAT_COLS = [
    "SEXO",
    "EDAD_SIMPLIFICADA",
    "PREVISION_GRUPAL",
    "DIAG1_GRUPO",
    "TIENE_DIAG2",
    "PERTENENCIA_ESTABLECIMIENTO_SALUD"
]

df_clean = df.filter(
    (col("DIAS_ESTADA") > 0) & (col("DIAS_ESTADA") <= 14) &
    (col("CONDICION_EGRESO") == 1) &
    (col("SEXO") != "*") &
    (col("GLOSA_PREVISION") != "*")
).filter(
    ~((col("SEXO") == "1") & (col("DIAG1").startswith("O")))
)

df_ml = df_clean.withColumn(
    "PREVISION_GRUPAL",
    when(col("GLOSA_PREVISION").isin("FONASA", "ISAPRE"), col("GLOSA_PREVISION"))
    .otherwise("OTRAS_O_FF.AA")
).withColumn(
    "EDAD_SIMPLIFICADA",
    when(col("GRUPO_EDAD").rlike("MES|DIA|menor|1 A 4|5 A 9|10 A 14"), "PEDIATRICO")
    .when(col("GRUPO_EDAD").rlike("15 A 19|20 A 24|25 A 29"), "JOVEN")
    .when(col("GRUPO_EDAD").rlike("30 A 34|35 A 39|40 A 44|45 A 49|50 A 54|55 A 59"), "ADULTO")
    .otherwise("ADULTO_MAYOR")
).withColumn(
    "DIAG1_GRUPO", substring(col("DIAG1"), 1, 1)
).withColumn(
    "TIENE_DIAG2",
    when(col("DIAG2").isNull() | (col("DIAG2") == "") | (col("DIAG2") == "SIN_DIAG2"), "NO")
    .otherwise("SI")
).withColumn(
    "label", col("DIAS_ESTADA").cast("double")
)

df_prepared_input = df_ml.select(CAT_COLS + ["label"]).dropna().sample(fraction=FRACTION_SAMPLE, seed=SEED)
indexers = [StringIndexer(inputCol=c, outputCol=c+"_idx", handleInvalid="keep") for c in CAT_COLS]
encoders = [OneHotEncoder(inputCol=c+"_idx", outputCol=c+"_ohe") for c in CAT_COLS]

assembler = VectorAssembler(
    inputCols=[c+"_ohe" for c in CAT_COLS],
    outputCol="features_unscaled"
)

scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withStd=True,
    withMean=False
)

pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])
pipeline_model = pipeline.fit(df_prepared_input)
df_final = pipeline_model.transform(df_prepared_input)

train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=SEED)
train_data.cache()

Datos listos. Entrenamiento: 117728 | Prueba: 29542


In [ ]:
############################
### Evaluacion de modelo ###
############################

In [ ]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import abs, col, round
import pandas as pd

param_regul = 0.1

lasso_model = LinearRegression(featuresCol="features", labelCol="label", regParam=param_regul, elasticNetParam=1.0).fit(train_data)
ridge_model = LinearRegression(featuresCol="features", labelCol="label", regParam=param_regul, elasticNetParam=0.0).fit(train_data)
elastic_net = LinearRegression(featuresCol="features", labelCol="label", regParam=param_regul, elasticNetParam=0.5).fit(train_data)

eval_rmse = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
eval_r2 = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="r2")
eval_mae = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="mae") # <--- NUEVO

modelos = [("Lasso (L1)", lasso_model), ("Ridge (L2)", ridge_model), ("Elastic Net", elastic_net)]
resultados = []

mejor_rmse = 999
mejores_predicciones = None

total_test = test_data.count()

for nombre, mod in modelos:
    predicciones = mod.transform(test_data)
    rmse = eval_rmse.evaluate(predicciones)
    r2 = eval_r2.evaluate(predicciones)
    mae = eval_mae.evaluate(predicciones)
    aciertos_tol = predicciones.filter(abs(col("label") - col("prediction")) <= 2.0).count()
    pct_acierto = (aciertos_tol / total_test) * 100

    resultados.append([nombre, rmse, r2, mae, pct_acierto])

    if rmse < mejor_rmse:
        mejor_rmse = rmse
        mejores_predicciones = predicciones

comparar = pd.DataFrame(resultados, columns=["Modelo", "RMSE", "R2", "MAE (Días)", "% Acierto (+/- 2 días)"])
print("\n--- TABLA COMPARATIVA DE MODELOS FINAL ---")
display(comparar.sort_values(by="RMSE", ascending=True).round(4))

print("\n--- EJEMPLO REAL DE PREDICCIONES (Mejor Modelo) ---")
(mejores_predicciones
 .select("SEXO", "EDAD_SIMPLIFICADA", "DIAG1_GRUPO", "label", round("prediction", 1).alias("Prediccion_Redondeada"))
 .withColumnRenamed("label", "Dias_Reales")
 .sample(fraction=0.01)
 .limit(10)
 .show()
)


--- TABLA COMPARATIVA DE MODELOS FINAL ---


,Modelo,RMSE,R2,MAE (Días),% Acierto (+/- 2 días)
1,Ridge (L2),2.7257,0.1630,1.9795,63.3776
2,Elastic Net,2.7356,0.1569,1.9937,63.5874
0,Lasso (L1),2.7559,0.1443,2.0175,62.3553



--- EJEMPLO REAL DE PREDICCIONES (Mejor Modelo) ---
+----+-----------------+-----------+-----------+---------------------+
|SEXO|EDAD_SIMPLIFICADA|DIAG1_GRUPO|Dias_Reales|Prediccion_Redondeada|
+----+-----------------+-----------+-----------+---------------------+
|   1|           ADULTO|          D|        3.0|                  3.6|
|   1|           ADULTO|          D|        5.0|                  3.6|
|   1|           ADULTO|          E|        4.0|                  3.8|
|   1|           ADULTO|          E|        6.0|                  3.8|
|   1|           ADULTO|          G|        1.0|                  4.0|
|   1|           ADULTO|          H|        1.0|                  1.3|
|   1|           ADULTO|          J|        3.0|                  4.4|
|   1|           ADULTO|          J|        3.0|                  4.4|
|   1|           ADULTO|          J|        4.0|                  4.4|
|   1|           ADULTO|          K|        1.0|                  2.0|
+----+-----------------+

In [ ]:
# TABLA TOP 10
def obtener_nombres_variables(df_prep, nombre_columna_features="features_unscaled"):
    attrs = df_prep.schema[nombre_columna_features].metadata["ml_attr"]["attrs"]
    nombres = []
    for attr_type in ["numeric", "binary"]:
        if attr_type in attrs:
            for attr in attrs[attr_type]:
                nombres.append({"idx": attr["idx"], "name": attr["name"]})
    nombres = sorted(nombres, key=lambda x: x["idx"])
    return [x["name"] for x in nombres]
nombres_columnas = obtener_nombres_variables(df_prepared)

ranking_lasso = pd.DataFrame(lasso_model.coefficients.toArray(), columns=["Coeficientes"])
ranking_lasso["Variables"] = nombres_columnas
ranking_lasso["Importancia_Absoluta"] = ranking_lasso["Coeficientes"].abs()
top_10_lasso = ranking_lasso.sort_values(by="Importancia_Absoluta", ascending=False).head(10)

ranking_ridge = pd.DataFrame(ridge_model.coefficients.toArray(), columns=["Coeficientes"])
ranking_ridge["Variables"] = nombres_columnas
ranking_ridge["Importancia_Absoluta"] = ranking_ridge["Coeficientes"].abs()
top_10_ridge = ranking_ridge.sort_values(by="Importancia_Absoluta", ascending=False).head(10)

ranking_elastic = pd.DataFrame(elastic_net.coefficients.toArray(), columns=["Coeficientes"])
ranking_elastic["Variables"] = nombres_columnas
ranking_elastic["Importancia_Absoluta"] = ranking_elastic["Coeficientes"].abs()
top_10_elastic = ranking_elastic.sort_values(by="Importancia_Absoluta", ascending=False).head(10)

print("--- TOP 10 VARIABLES: LASSO ---")
display(top_10_lasso)

print("--- TOP 10 VARIABLES: RIDGE ---")
display(top_10_ridge)

print("--- TOP 10 VARIABLES: ELASTIC NET ---")
display(top_10_elastic)




--- TOP 10 VARIABLES: LASSO ---


,Coeficientes,Variables,Importancia_Absoluta
3,0.430220,EDAD_SIMPLIFICADA_ohe_ADULTO_MAYOR,0.430220
33,0.353321,PERTENENCIA_ESTABLECIMIENTO_SALUD_ohe_Pertenec...,0.353321
34,-0.353321,PERTENENCIA_ESTABLECIMIENTO_SALUD_ohe_No Perte...,0.353321
13,0.189170,DIAG1_GRUPO_ohe_I,0.189170
22,0.179614,DIAG1_GRUPO_ohe_F,0.179614
19,-0.106131,DIAG1_GRUPO_ohe_Z,0.106131
15,-0.084569,DIAG1_GRUPO_ohe_M,0.084569
10,-0.056937,DIAG1_GRUPO_ohe_O,0.056937
11,0.052202,DIAG1_GRUPO_ohe_J,0.052202
20,0.033333,DIAG1_GRUPO_ohe_T,0.033333


--- TOP 10 VARIABLES: RIDGE ---


,Coeficientes,Variables,Importancia_Absoluta
33,0.381137,PERTENENCIA_ESTABLECIMIENTO_SALUD_ohe_Pertenec...,0.381137
34,-0.381137,PERTENENCIA_ESTABLECIMIENTO_SALUD_ohe_No Perte...,0.381137
3,0.358321,EDAD_SIMPLIFICADA_ohe_ADULTO_MAYOR,0.358321
22,0.270098,DIAG1_GRUPO_ohe_F,0.270098
5,-0.245253,EDAD_SIMPLIFICADA_ohe_PEDIATRICO,0.245253
13,0.225997,DIAG1_GRUPO_ohe_I,0.225997
19,-0.204529,DIAG1_GRUPO_ohe_Z,0.204529
15,-0.189889,DIAG1_GRUPO_ohe_M,0.189889
11,0.163671,DIAG1_GRUPO_ohe_J,0.163671
4,-0.148186,EDAD_SIMPLIFICADA_ohe_JOVEN,0.148186


--- TOP 10 VARIABLES: ELASTIC NET ---


,Coeficientes,Variables,Importancia_Absoluta
3,0.433378,EDAD_SIMPLIFICADA_ohe_ADULTO_MAYOR,0.433378
33,0.368509,PERTENENCIA_ESTABLECIMIENTO_SALUD_ohe_Pertenec...,0.368509
34,-0.368509,PERTENENCIA_ESTABLECIMIENTO_SALUD_ohe_No Perte...,0.368509
22,0.230247,DIAG1_GRUPO_ohe_F,0.230247
13,0.218762,DIAG1_GRUPO_ohe_I,0.218762
19,-0.149846,DIAG1_GRUPO_ohe_Z,0.149846
15,-0.128958,DIAG1_GRUPO_ohe_M,0.128958
11,0.114100,DIAG1_GRUPO_ohe_J,0.114100
20,0.082226,DIAG1_GRUPO_ohe_T,0.082226
10,-0.081746,DIAG1_GRUPO_ohe_O,0.081746
